In [3]:
import os
import pandas as pd

print("Iniciando la construcción DEFINITIVA del Tablón Analítico (Capa Oro)...")

# Cargar la capa Plata
print("Cargando Clima, Demanda y Calendario Laboral...")
df_clima = pd.read_parquet("../data_lake/plata/aemet_limpio.parquet")
df_demanda = pd.read_parquet("../data_lake/plata/demanda_limpia.parquet")
df_festivos = pd.read_parquet("../data_lake/plata/festivos_limpios.parquet")

# Homogeneizar fechas
df_clima['fecha'] = pd.to_datetime(df_clima['fecha'])
df_demanda['fecha'] = pd.to_datetime(df_demanda['fecha'])
df_festivos['fecha'] = pd.to_datetime(df_festivos['fecha'])

print("Limpiando formatos de temperatura...")
columnas_temp = ['tmax', 'tmin', 'tmed']
for col in columnas_temp:
    if col in df_clima.columns:
        df_clima[col] = df_clima[col].astype(str).str.replace(',', '.')
        df_clima[col] = pd.to_numeric(df_clima[col], errors='coerce')

# Reducir la granularidad del clima a Media Nacional
print("Calculando media climática nacional...")
df_clima_nacional = df_clima.groupby('fecha')[['tmax', 'tmin', 'tmed']].mean().reset_index()

# Cruce de vías
print("Fusionando las tres fuentes de datos...")

# Cruzamos Clima y Demanda (Solo los días que existan en ambos)
df_oro = pd.merge(df_clima_nacional, df_demanda, on='fecha', how='inner')

# Le añadimos el calendario de festivos (Usamos 'left' para que simplemente pegue el dato)
df_oro = pd.merge(df_oro, df_festivos, on='fecha', how='left')

# Limpieza final y guardado
df_oro['es_festivo'] = df_oro['es_festivo'].fillna(0).astype(int)
df_oro = df_oro.sort_values('fecha').reset_index(drop=True)

ruta_oro = "../data_lake/oro/dataset_analitico.parquet"
df_oro.to_parquet(ruta_oro, index=False, engine='pyarrow')

print("\n=======================================================")
print(f"¡CAPA ORO TOTALMENTE COMPLETADA Y ENRIQUECIDA!")
print(f"Columnas finales: {list(df_oro.columns)}")
print("=======================================================")

Iniciando la construcción DEFINITIVA del Tablón Analítico (Capa Oro)...
Cargando Clima, Demanda y Calendario Laboral...
Limpiando formatos de temperatura...
Calculando media climática nacional...
Fusionando las tres fuentes de datos...

¡CAPA ORO TOTALMENTE COMPLETADA Y ENRIQUECIDA!
Columnas finales: ['fecha', 'tmax', 'tmin', 'tmed', 'demanda_mwh', 'es_festivo']
